LLM Memory Types
- Mainly memory is divided into two parts short-term and long-term  
**Short-term :**
1. Buffer Memory
2. Window Memory
3. Summary Memory
4. SummaryBufferMemory  
**Long-term :**
1. Entity Memory
2. Vector Store Memory


---

In [1]:
from google.colab import userdata
import os
os.environ['GOOGLE_API_KEY'] = userdata.get('gemini_key')

In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    google_genai

1. **Buffer Memory-Full history, no trimming**  
- The simplest form. Every single message is stored and passed to the LLM every time.
- Use when: conversations are short, token limits aren't a concern, and you need the LLM to have full context.

In [13]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI

# model
llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    temperature=0
)
# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),   # memory goes here
    ("human", "{input}"),
])
#LCEL
chain = prompt | llm

# In-memory store — one history object per session_id
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)


In [14]:
resp1 = chain_with_memory.invoke(
    {"input": "Hi, my name is ruDra."},
    config={"configurable": {"session_id": "user_1"}}
)
resp2 = chain_with_memory.invoke(
    {"input": "What's my name?"},
    config={"configurable": {"session_id": "user_1"}}
)
print(resp2.content)

Your name is Rudra.


___

2. **Window Memory**
- Keeps a sliding window of the last k human+AI exchanges and discards older ones.
- Use when: conversations are long but recent context is what matters most (e.g., a support chatbot).

In [7]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from typing import List

class WindowChatHistory(BaseChatMessageHistory):
    """Keeps only the last k messages."""

    def __init__(self, k: int = 6):   # k=6 means 3 human + 3 AI turns
        self.messages: List[BaseMessage] = []
        self.k = k

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
        self.messages = self.messages[-self.k:]   # trim to window

    def clear(self) -> None:
        self.messages = []

window_store = {}

def get_window_history(session_id: str) -> WindowChatHistory:
    if session_id not in window_store:
        window_store[session_id] = WindowChatHistory(k=6)
    return window_store[session_id]

chain_with_window = RunnableWithMessageHistory(
    chain,
    get_window_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [15]:
chain_with_window.invoke({"input": "My favourite food is biryani."},
                         config={"configurable": {"session_id": "user_2"}})
chain_with_window.invoke({"input": "I live in Hyderabad."},
                         config={"configurable": {"session_id": "user_2"}})
chain_with_window.invoke({"input": "I love cricket."},
                         config={"configurable": {"session_id": "user_2"}})
chain_with_window.invoke({"input": "I enjoy reading."},
                         config={"configurable": {"session_id": "user_2"}})

AIMessage(content="That's a wonderful addition to your interests! Reading is such a rich and rewarding hobby.\n\nSo, now we know you're a Hyderabadi who loves biryani, cricket, and reading. That paints a pretty well-rounded picture!\n\nWhat kind of books or genres do you typically enjoy reading? Are you into fiction, non-fiction, thrillers, history, fantasy, or something else entirely?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dda6f-bd21-7d71-988c-8ea0fe2da77f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 309, 'output_tokens': 244, 'total_tokens': 553, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 158}})

In [18]:
resp = chain_with_window.invoke({"input": "I  live in?."},
                         config={"configurable": {"session_id": "user_2"}})
print(resp.content)

Ah, I already know this one!

You live in **Hyderabad**. 😊

We established that right at the beginning! It's your hometown, and you love it there.


---

3. **Summary Memory:**
- Instead of dropping old messages, it uses an LLM to summarize the conversation so far. The prompt receives a concise summary, not raw history.
- Use when: you need long-term context but can't afford to pass the full history each time (e.g., a research assistant session).

In [20]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

summarizer = ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0)

class SummaryChatHistory(BaseChatMessageHistory):
    def __init__(self, max_messages: int = 6):
        self.messages: List[BaseMessage] = []
        self.summary: str = ""
        self.max_messages = max_messages

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages:
            self._summarize_and_trim()

    def _summarize_and_trim(self):
        old = self.messages[:-4]   # everything except last 4
        recent = self.messages[-4:]
        text = "\n".join(f"{m.type}: {m.content}" for m in old)
        result = summarizer.invoke(
            f"Summarize this conversation concisely:\n{text}"
        )
        self.summary = result.content
        self.messages = [SystemMessage(content=f"Summary: {self.summary}")] + recent

    def clear(self):
        self.messages = []
        self.summary = ""

summary_store = {}

def get_summary_history(session_id):
    if session_id not in summary_store:
        summary_store[session_id] = SummaryChatHistory(max_messages=6)
    return summary_store[session_id]

chain_with_summary = RunnableWithMessageHistory(
    chain,
    get_summary_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [22]:
chain_with_summary.invoke({"input": "Hi, my name is Rudra. I am building a crypto chatbot."},
                         config={"configurable": {"session_id": "user_3"}})

AIMessage(content="Hi Rudra! That's a really interesting project! Building a crypto chatbot sounds like a fantastic idea, especially given the dynamic nature of the crypto world.\n\nTo help me understand better and offer the most relevant insights, could you tell me a bit more about what you envision for your chatbot? For example:\n\n*   **What kind of functionalities are you planning to include?** (e.g., real-time price updates, news aggregation, educational content, portfolio tracking, technical analysis, DeFi explanations, NFT insights, etc.)\n*   **Who is your target audience?** (e.g., beginners, experienced traders, developers)\n*   **What problem are you hoping to solve with it?**\n*   **Are there any specific challenges you're anticipating or have already encountered?**\n\nI'm excited to hear more about it and happy to help brainstorm ideas, discuss potential challenges, or even delve into specific technical aspects if you'd like!", additional_kwargs={}, response_metadata={'fini

In [23]:
chain_with_summary.invoke({"input": "i like biryani"},
                         config={"configurable": {"session_id": "user_3"}})

AIMessage(content="Haha, excellent choice, Rudra! Biryani is a universally loved dish – truly a culinary masterpiece.\n\nIt's always good to have a delicious thought in mind!\n\nSo, back to your crypto chatbot project – were you thinking about how to integrate some of those functionalities we discussed, or perhaps you had another question about it? I'm ready when you are! 😊", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dda81-5f4b-75a1-869b-a8bc296d88dc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 228, 'output_tokens': 299, 'total_tokens': 527, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 222}})

In [24]:
chain_with_summary.invoke({"input": "My habits include playing chess , reading books"},
                         config={"configurable": {"session_id": "user_3"}})

AIMessage(content='Those are fantastic habits, Rudra!\n\n*   **Chess** is brilliant for strategic thinking, pattern recognition, and foresight – skills that are surprisingly relevant in the world of crypto trading and understanding market dynamics!\n*   **Reading books** is, of course, invaluable for continuous learning, which is absolutely essential in the fast-evolving crypto space.\n\nIt sounds like you have a great foundation for analytical thinking.\n\nSpeaking of analysis and learning, how are you thinking about incorporating educational content or market analysis features into your crypto chatbot? Or perhaps you had another thought about your project?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dda82-5ade-76b2-885c-38f26ff7c950-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 315, 'output_tokens': 292, 'total_tokens': 607, 'in

In [25]:
resp = chain_with_summary.invoke({"input": "can you summarize the details up to now"},
                         config={"configurable": {"session_id": "user_3"}})
print(resp.content)

Okay, Rudra, here's a quick summary of what we've covered so far:

1.  **Your Name:** Rudra
2.  **Your Project:** You are building a **crypto chatbot**.
3.  **Your Preferences:** You like **Biryani**.
4.  **Your Habits:** You enjoy playing **chess** and **reading books**.

We've also touched upon potential functionalities for your chatbot and my offer to help brainstorm, but the core details about *you* are the points above.

What's next on your mind for the chatbot?
